<a id="markov-chain"></a>

$$
\text{马尔可夫链 (Markov Chain): 状态随时间随机转移, 并且 "未来只和现在有关, 和过去无关"} \\ \quad \\
\text{马尔可夫性 (Markov Property): } P(X_{n+1} = j \mid X_n = i, X_{n-1}, \cdots, X_0) = P(X_{n+1} = j \mid X_n = i) = p_{ij} \\
\text{(时齐 Time-Homogeneous: } p_{ij} \text{ 不随 n 变化)} \\ \quad \\
\text{转移矩阵 (Transition Matrix): } P = [p_{ij}] \text{, 第 i 行是 "从状态 i 出发, 下一步去各个状态的概率"} \\
p_{ij} \geq 0, \quad \sum_j p_{ij} = 1 \text{ (每一行和为 1, 随机矩阵 Stochastic Matrix)} \\ \quad \\
\text{状态分布的演化: 设 } \pi^{(n)} \text{ 是第 n 步处于各状态的概率 (行向量), 由全概率公式} \\
\pi_j^{(n+1)} = P(X_{n+1} = j) = \sum_i P(X_n = i) \, P(X_{n+1} = j \mid X_n = i) = \sum_i \pi_i^{(n)} p_{ij} \\
\pi^{(n+1)} = \pi^{(n)} P \implies \pi^{(n)} = \pi^{(0)} P^n \\ \quad \\
\text{切普曼-科尔莫戈罗夫方程 (Chapman-Kolmogorov): n 步转移概率 } p_{ij}^{(n)} = P(X_n = j \mid X_0 = i) \\
p_{ij}^{(m+n)} = \sum_k P(X_m = k \mid X_0 = i) P(X_{m+n} = j \mid X_m = k) = \sum_k p_{ik}^{(m)} p_{kj}^{(n)} \text{ (对中间状态用全概率公式)} \\
P^{(m+n)} = P^{(m)} P^{(n)} \implies P^{(n)} = P^n \text{ (n 步转移矩阵就是矩阵的 n 次方)}
$$

数学基础: [全概率公式与贝叶斯](../math_basics/statistics.ipynb#bayes) · [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

# 天气: 0 = 晴, 1 = 多云, 2 = 雨
states = ['sunny', 'cloudy', 'rainy']
P_w = np.array([[0.7, 0.2, 0.1],
                [0.3, 0.4, 0.3],
                [0.2, 0.3, 0.5]])
print(f'每行和 = {P_w.sum(axis=1)}')

pi0 = np.array([1.0, 0.0, 0.0])                 # 今天晴天
for n in [1, 2, 3, 10, 30]:
    print(f'{n:>2} 天后的分布 = {np.round(pi0 @ np.linalg.matrix_power(P_w, n), 6)}')

# Chapman-Kolmogorov: P^(2+3) == P^2 P^3, 并用双重求和逐项验证
P2, P3 = np.linalg.matrix_power(P_w, 2), np.linalg.matrix_power(P_w, 3)
P5_sum = np.array([[sum(P2[i, k] * P3[k, j] for k in range(3)) for j in range(3)] for i in range(3)])
print(f'P^5 == P^2 P^3 (逐项求和) ? {np.allclose(np.linalg.matrix_power(P_w, 5), P5_sum)}')

# 模拟 10 万条链, 3 天后的分布
rng = np.random.default_rng(0)
n_chains = 100000
x = np.zeros(n_chains, dtype=int)
for _ in range(3):
    u = rng.random(n_chains)
    x = (u[:, None] > np.cumsum(P_w[x], axis=1)).sum(axis=1)   # 按当前状态那一行的概率抽下一个状态
print(f'模拟 3 天后的分布 = {np.bincount(x, minlength=3) / n_chains}')

<a id="stationary-distribution"></a>

$$
\text{平稳分布 (Stationary Distribution): 转移一步后分布不变} \\
\pi = \pi P, \quad \sum_i \pi_i = 1 \\ \quad \\
\text{和特征值的关系: 两边转置 } P^T \pi^T = \pi^T \text{, 所以 } \pi^T \text{ 是 } P^T \text{ 的特征值为 1 的特征向量 (也叫 P 的左特征向量)} \\ \quad \\
\text{1. 为什么一定有特征值 1: 每行和为 1 } \implies P \mathbf{1} = \mathbf{1} \text{, 所以 1 是 P 的特征值;} \\
P \text{ 和 } P^T \text{ 的特征多项式相同 } (\det(P^T - \lambda I) = \det((P - \lambda I)^T) = \det(P - \lambda I)) \text{, 所以 1 也是 } P^T \text{ 的特征值} \\
\text{2. 为什么所有特征值 } |\lambda| \leq 1 \text{: 设 } P v = \lambda v \text{, 取 } |v_i| \text{ 最大的分量 i} \\
|\lambda| |v_i| = \left|\sum_j p_{ij} v_j\right| \leq \sum_j p_{ij} |v_j| \leq \sum_j p_{ij} |v_i| = |v_i| \implies |\lambda| \leq 1 \\ \quad \\
\text{3. 收敛速度: 设 P 可对角化 } P = V \Lambda V^{-1} \text{, 则 } P^n = V \Lambda^n V^{-1} \\
\text{特征值 1 对应的部分保持不变, 其余 } |\lambda_k| < 1 \text{ 的部分按 } |\lambda_k|^n \text{ 衰减} \\
\pi^{(n)} - \pi \sim |\lambda_2|^n \quad (\lambda_2 \text{ 是模第二大的特征值, } |\lambda_2| \text{ 越小收敛越快}) \\ \quad \\
\text{4. 收敛条件: 不可约 (Irreducible, 任意两个状态互相可达) + 非周期 (Aperiodic)} \\
\text{反例: } P = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} \text{ 特征值 } \pm 1 \text{, } |\lambda_2| = 1 \text{, 分布永远在两个状态之间来回跳, 不收敛} \\ \quad \\
\text{求法: 解线性方程组 } (P^T - I) \pi^T = 0 \text{, 这 n 个方程线性相关 (秩 n-1), 用 } \sum \pi_i = 1 \text{ 替换其中一个}
$$

数学基础: [特征值与特征向量](../math_basics/linear_algebra.ipynb#eigen) · [对角化与 A^k](../math_basics/linear_algebra.ipynb#diagonalization) · [矩阵的秩](../math_basics/linear_algebra.ipynb#rank)

In [ ]:
# 方法 1: P^T 的特征值 1 对应的特征向量
eigval, eigvec = np.linalg.eig(P_w.T)
k1 = np.argmin(np.abs(eigval - 1))
pi_eig = np.real(eigvec[:, k1])
pi_eig = pi_eig / pi_eig.sum()
print(f'P^T 的特征值 = {np.round(eigval, 6)}')
print(f'方法 1 (特征向量): pi = {np.round(pi_eig, 6)}')

# 方法 2: 解线性方程组, 最后一个方程换成 sum = 1
A_lin = P_w.T - np.eye(3)
A_lin[-1] = 1
b_lin = np.array([0, 0, 1.0])
pi_lin = np.linalg.solve(A_lin, b_lin)
print(f'方法 2 (线性方程组): pi = {np.round(pi_lin, 6)}')

# 方法 3: P^n 的每一行都收敛到 pi
print(f'方法 3 (P^100 的每一行): \n{np.round(np.linalg.matrix_power(P_w, 100), 6)}')

# 方法 4: 一条很长的链中, 各状态出现的频率
rng = np.random.default_rng(1)
n_long = 200000
u = rng.random(n_long)
cum = np.cumsum(P_w, axis=1)
chain = np.zeros(n_long, dtype=int)
for t in range(1, n_long):
    chain[t] = np.searchsorted(cum[chain[t - 1]], u[t])
print(f'方法 4 (长链频率): pi = {np.round(np.bincount(chain) / n_long, 4)}')
print(f'pi P == pi ? {np.allclose(pi_lin @ P_w, pi_lin)}, 特征向量 == 线性方程组 ? {np.allclose(pi_eig, pi_lin)}')

# 收敛速度 ~ |lambda_2|^n
lam2 = np.sort(np.abs(eigval))[-2]
ns = np.arange(1, 25)
errors = [np.abs(pi0 @ np.linalg.matrix_power(P_w, n) - pi_lin).sum() for n in ns]
plt.figure(figsize=(4, 3))
plt.semilogy(ns, errors, 'o-', label='|pi(n) - pi|')
plt.semilogy(ns, errors[0] * lam2**(ns - 1), 'r--', label=f'|lambda_2|^n, |lambda_2| = {lam2:.3f}')
plt.xlabel('n')
plot_nicely()
plt.show()

# 周期链不收敛
P_periodic = np.array([[0.0, 1.0], [1.0, 0.0]])
periodic_dists = np.array([np.array([1.0, 0.0]) @ np.linalg.matrix_power(P_periodic, n) for n in range(4)])
print(f'周期链的特征值 = {np.linalg.eigvals(P_periodic)}')
print(f'前 4 步的分布 (来回跳, 不收敛) = \n{periodic_dists}')

<a id="absorbing-chain"></a>

$$
\text{吸收马尔可夫链 (Absorbing Markov Chain): 有些状态一旦进入就不再离开 } (p_{ii} = 1) \\
\text{把状态排成 "暂态 (Transient) 在前, 吸收态 (Absorbing) 在后" 的标准形式:} \\
P = \begin{bmatrix} Q & R \\ 0 & I \end{bmatrix} \quad Q \text{: 暂态之间的转移, R: 暂态到吸收态的转移} \\ \quad \\
\text{1. 基本矩阵 (Fundamental Matrix): } N = I + Q + Q^2 + Q^3 + \cdots = (I - Q)^{-1} \\
\text{推导: } (Q^n)_{ij} \text{ 是从暂态 i 出发, 第 n 步在暂态 j 的概率, 对 n 求和就是访问 j 的期望次数} \\
\text{几何级数: } (I - Q)(I + Q + \cdots + Q^{n}) = I - Q^{n+1} \text{, 只要 } Q^n \to 0 \text{ (最终一定会被吸收) 就有 } \sum_{k=0}^\infty Q^k = (I - Q)^{-1} \\
\text{(和幂零矩阵 } (I - N)^{-1} = I + N + \cdots + N^{k-1} \text{ 是同一个公式, 区别是这里级数无穷长但收敛)} \\ \quad \\
\text{2. 被吸收前的期望步数: } t = N \mathbf{1} \text{ (把访问每个暂态的次数加起来)} \\
\text{3. 被各个吸收态吸收的概率: } B = N R \\
\text{推导: 从 i 出发最终被 k 吸收 = 先在暂态之间走若干步到达某个暂态 j, 再一步从 j 进入 k: } B_{ik} = \sum_j N_{ij} R_{jk} \\ \quad \\
\text{例: 赌徒破产 (Gambler's Ruin), 本金 0 到 4 元, 每次以 1/2 的概率赢或输 1 元, 0 元和 4 元时停止} \\
\text{从 i 元开始, 最终赢到 4 元的概率 } = \frac{i}{4} \text{ (公平游戏, 期望财富不变: } 4 \cdot p + 0 \cdot (1 - p) = i \text{)} \\
\text{期望局数 } = i(4 - i)
$$

数学基础: [幂零矩阵与几何级数](../math_basics/linear_algebra.ipynb#nilpotent) · [矩阵的逆](../math_basics/linear_algebra.ipynb#inverse) · [数学期望](../math_basics/statistics.ipynb#expectation)

In [ ]:
# 暂态: 1, 2, 3 元; 吸收态: 0 元, 4 元
Q = np.array([[0.0, 0.5, 0.0],
              [0.5, 0.0, 0.5],
              [0.0, 0.5, 0.0]])
R = np.array([[0.5, 0.0],
              [0.0, 0.0],
              [0.0, 0.5]])
N = np.linalg.inv(np.eye(3) - Q)
series = sum(np.linalg.matrix_power(Q, k) for k in range(200))
print(f'基本矩阵 N = \n{N}\n== I + Q + Q^2 + ... (200 项) ? {np.allclose(N, series)}')

t_steps = N @ np.ones(3)
B = N @ R
start = np.array([1, 2, 3])
print(f'期望局数 = {t_steps}, 理论 i(4 - i) = {start * (4 - start)}')
print(f'被 4 元吸收的概率 = {B[:, 1]}, 理论 i/4 = {start / 4}')

# 模拟
rng = np.random.default_rng(2)
for i in start:
    wins, lengths = 0, []
    for _ in range(20000):
        money, steps = i, 0
        while 0 < money < 4:
            money += 1 if rng.random() < 0.5 else -1
            steps += 1
        wins += money == 4
        lengths.append(steps)
    print(f'从 {i} 元开始: 模拟赢的概率 = {wins / 20000:.4f}, 模拟平均局数 = {np.mean(lengths):.4f}')

<a id="market-share"></a>

$$
\text{应用: 市场占有率预测} \\ \quad \\
\text{1. 从数据估计转移矩阵: 统计 "上一期用品牌 i, 这一期用品牌 j" 的人数 } n_{ij} \\
\hat{p}_{ij} = \frac{n_{ij}}{\sum_k n_{ik}} \quad (\text{这是最大似然估计: 从状态 i 出发的转移是一个多项分布,} \\
\text{对数似然 } \sum_j n_{ij} \ln p_{ij} \text{ 在 } \sum_j p_{ij} = 1 \text{ 约束下用拉格朗日乘子: } \frac{n_{ij}}{p_{ij}} = \lambda \implies p_{ij} \propto n_{ij} \text{)} \\
\text{2. 预测: } \pi^{(n)} = \pi^{(0)} P^n \\
\text{3. 长期占有率: 平稳分布 } \pi = \pi P \text{ (和现在的占有率无关)}
$$

数学基础: [最大似然估计](../math_basics/statistics.ipynb#mle) · [伯努利分布的 MLE (两个类别的情况)](../math_basics/statistics.ipynb#mle-bernoulli) · [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange)

In [ ]:
brands = ['A', 'B', 'C']
P_true = np.array([[0.80, 0.10, 0.10],
                   [0.15, 0.75, 0.10],
                   [0.20, 0.20, 0.60]])

# 模拟 3000 个顾客的两期购买记录, 再从计数估计转移矩阵
rng = np.random.default_rng(3)
share_now = np.array([0.4, 0.35, 0.25])
before = rng.choice(3, size=3000, p=share_now)
after = np.array([rng.choice(3, p=P_true[b]) for b in before])
counts = np.zeros((3, 3))
np.add.at(counts, (before, after), 1)
P_hat = counts / counts.sum(axis=1, keepdims=True)
print(f'转移计数 = \n{counts.astype(int)}')
print(f'估计的转移矩阵 = \n{np.round(P_hat, 4)}')

# 独立验证 MLE: 对第 0 行, 在概率单纯形上随机取点, 对数似然都不超过计数比例处的值
loglik = lambda p: np.sum(counts[0] * np.log(p))
random_rows = rng.dirichlet(np.ones(3), size=50000)
print(f'第 0 行: 计数比例的对数似然 = {loglik(P_hat[0]):.4f}, 5 万个随机点的最大值 = {max(loglik(p) for p in random_rows):.4f}')

shares = [share_now]
for _ in range(10):
    shares.append(shares[-1] @ P_hat)
shares = np.array(shares)
A_lin = P_hat.T - np.eye(3)
A_lin[-1] = 1
pi_market = np.linalg.solve(A_lin, [0, 0, 1.0])
print(f'下一期占有率 = {np.round(shares[1], 4)}, 长期占有率 = {np.round(pi_market, 4)}')

plt.figure(figsize=(4, 3))
for k, b in enumerate(brands):
    plt.plot(shares[:, k], 'o-', markersize=3, label=f'brand {b}')
    plt.axhline(pi_market[k], color=f'C{k}', linestyle='--', linewidth=0.8)
plt.xlabel('period')
plt.ylabel('market share')
plot_nicely()
plt.show()